# 03b - Strong Models
Goal: train Random Forest and LightGBM for binary and multi-class detection, handle class imbalance, tune hyperparameters and save the best multi-class model.

## 1. Setup
Load the processed train/test sets and split them into features (X) and labels (y). `FEATURE_COLS` from `src/config.py` fixes the column order, so the saved model gets the same input order in the demo and the Mininet test.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.config import RANDOM_STATE, FEATURE_COLS, LABEL_COLS

TRAIN_PATH = '../data/processed/train.parquet'
TEST_PATH  = '../data/processed/test.parquet'

print('RANDOM_STATE:', RANDOM_STATE)
print('Number of features:', len(FEATURE_COLS))
print('Label columns:', LABEL_COLS)

RANDOM_STATE: 42
Number of features: 62
Label columns: ['label_multi', 'label_binary']


In [2]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

print('Train:', train.shape)
print('Test: ', test.shape)

Train: (341633, 64)
Test:  (85409, 64)


In [3]:
X_train = train[FEATURE_COLS]
X_test  = test[FEATURE_COLS]

y_train_bin = train['label_binary']
y_test_bin  = test['label_binary']

y_train_multi = train['label_multi']
y_test_multi  = test['label_multi']

print('X_train:', X_train.shape, '| X_test:', X_test.shape)

X_train: (341633, 62) | X_test: (85409, 62)


Class distribution in train and test. Thanks to the stratified split, the percentages should be almost the same in both sets.

In [4]:
dist = pd.DataFrame({
    'train_count': y_train_multi.value_counts(),
    'train_%': (y_train_multi.value_counts(normalize=True) * 100).round(2),
    'test_count': y_test_multi.value_counts(),
    'test_%': (y_test_multi.value_counts(normalize=True) * 100).round(2),
})
dist

,train_count,train_%,test_count,test_%
label_multi,,,,
Benign,186996,54.74,46749,54.74
Dos,101205,29.62,25301,29.62
DDoS,40215,11.77,10054,11.77
Bot,7165,2.10,1792,2.10
Prob,6052,1.77,1513,1.77


In [5]:
print('Binary labels (0 = Benign, 1 = Attack), train %:')
print((y_train_bin.value_counts(normalize=True) * 100).round(2))

Binary labels (0 = Benign, 1 = Attack), train %:
label_binary
0    54.74
1    45.26
Name: proportion, dtype: float64


## 2. evaluate()
Copied unchanged from `notebooks/03_models.ipynb` (baseline-models branch, commit e343c28), so all models are measured in the same way.

It trains the model, measures training time and prediction time per flow (ms), prints the classification report and returns one row for the results table. The function always returns `imbalance='none'` and empty `notes`, so we set these fields on the returned row when needed. It does not return the predictions, so for confusion matrices we call `model.predict` again on the already trained model.

In [6]:
import time
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [7]:
def evaluate(model, name, task, X_tr, y_tr, X_te, y_te):
    # 1) train and measure time
    t0 = time.perf_counter()
    model.fit(X_tr,y_tr)
    train_time = time.perf_counter() - t0

    # 2) predict and measure time
    t0 = time.perf_counter()
    y_pred = model.predict(X_te)
    predict_time = time.perf_counter() - t0
    predict_ms = predict_time / len(X_te) * 1000 #converts seconds to ms

    # 3) show the detailed report
    print(f"=== {name} | {task} ===")
    print(classification_report(y_te,y_pred,digits=4))

    # 4) return one row for results.csv
    return{
        "model":name,
        "task":task,
        "imbalance":"none",
        "macro_f1":f1_score(y_te,y_pred,average="macro"),
        "accuracy":accuracy_score(y_te,y_pred),
        "train_time_s":round(train_time,2),
        "predict_ms_per_flow":round(predict_ms,5),
        "notes":"",

    }


In [8]:
results = []  # one dict per model, written to reports/results_strong.csv at the end

## 3. Random Forest
Random Forest trains many decision trees, each on a bootstrap sample of the rows and a random subset of features at each split, and takes a majority vote. Averaging many different trees reduces overfitting compared to a single tree.

No scaler: a tree splits on thresholds such as `Flow Duration > 1000`, so the scale of a feature does not change which split is chosen. Default hyperparameters are used here (100 trees); tuning comes in section 6. `n_jobs=-1` uses all CPU cores.

In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
rf_bin = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

row = evaluate(rf_bin, "RandomForest", "binary",
               X_train, y_train_bin, X_test, y_test_bin)
results.append(row)
row

=== RandomForest | binary ===
              precision    recall  f1-score   support

           0     0.9213    0.9199    0.9206     46749
           1     0.9033    0.9049    0.9041     38660

    accuracy                         0.9131     85409
   macro avg     0.9123    0.9124    0.9124     85409
weighted avg     0.9131    0.9131    0.9131     85409



{'model': 'RandomForest',
 'task': 'binary',
 'imbalance': 'none',
 'macro_f1': 0.9123512667967151,
 'accuracy': 0.9131239096582328,
 'train_time_s': 11.12,
 'predict_ms_per_flow': 0.00186,
 'notes': ''}

In [11]:
rf_multi = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

row = evaluate(rf_multi, "RandomForest", "multi",
               X_train, y_train_multi, X_test, y_test_multi)
results.append(row)
row

=== RandomForest | multi ===


              precision    recall  f1-score   support

      Benign     0.9195    0.9212    0.9203     46749
         Bot     0.2926    0.2896    0.2911      1792
        DDoS     0.5634    0.5651    0.5643     10054
         Dos     0.9106    0.9096    0.9101     25301
        Prob     0.5521    0.5254    0.5384      1513

    accuracy                         0.8556     85409
   macro avg     0.6476    0.6422    0.6448     85409
weighted avg     0.8553    0.8556    0.8554     85409



{'model': 'RandomForest',
 'task': 'multi',
 'imbalance': 'none',
 'macro_f1': 0.6448476159595093,
 'accuracy': 0.8555889894507605,
 'train_time_s': 14.59,
 'predict_ms_per_flow': 0.00339,
 'notes': ''}